# 00b · Ceuta como validación externa

Ceuta es un sistema eléctrico aislado a menos de 40 km de Tetuán, con la demanda publicada por Red Eléctrica
(REE) cada 5 minutos desde mayo de 2019. Con varios años de datos se puede responder lo que Tetuán, con solo
2017, no permite: **cuánto mejora el pronóstico a una hora cuando el modelo ya vio uno, dos o tres veranos**,
y qué tan pesimista es la cota de verano que usamos en Tetuán.

Este notebook corre entero en Colab: baja los datos, arma las capas en el Drive y ejecuta la validación.

| Paso | Qué hace | Tiempo aproximado |
|---|---|---|
| Bronze | Una consulta a REE por día (unos 2,700 días, unos 105 MB) | ~1 h la primera vez; después, solo los días nuevos |
| Clima | Una consulta a Open-Meteo (ERA5 horario) | Segundos |
| Silver | Hora UTC cada 10 minutos, en kW, con el clima y una auditoría | ~1 min |
| Validación | Entrenar con 0 a 3 veranos y evaluar julio y agosto de 2020 a 2026 | ~20-40 min |

Si la sesión se corta, vuelve a correr todo: el bronze retoma en el mes en que quedó.

**Uso de los datos:** el aviso legal de REE permite usos académicos y no comerciales que citen la fuente.
Los datos quedan en tu Drive y no se suben a ningún repositorio.

## Instalación

La validación usa scripts de `experimentos/`, así que se clona el repo además de instalar el paquete.

In [ ]:
import os

RAMA = "develop"
if not os.path.exists("tetouan-power-forecast"):
    !git clone -q -b {RAMA} https://github.com/R0SEWT/tetouan-power-forecast
%cd tetouan-power-forecast
!git pull -q
!pip install -q -e .

import sys

# El kernel que ya corre no lee el .pth de la instalación editable: se agrega el código a mano
sys.path[:0] = [os.path.abspath("src"), os.path.abspath(".")]

In [ ]:
from tetouan_power_forecast.medallon import usar_drive

raiz = usar_drive()  # las capas de Ceuta quedan en la misma carpeta, dentro de ceuta/

## Bronze: la demanda de REE, día por día

In [ ]:
from tetouan_power_forecast.ceuta import bronze_ceuta

bronze_ceuta()

## Clima: ERA5 para Ceuta

In [ ]:
from tetouan_power_forecast.ceuta import clima_ceuta

clima_ceuta()

## Silver: cada 10 minutos, en kW, con su auditoría

In [ ]:
from IPython.display import Markdown, display

from tetouan_power_forecast.ceuta import SERIE, cargar_silver_ceuta, silver_ceuta

ruta = silver_ceuta()
display(Markdown(ruta.with_name("auditoria.md").read_text()))
ceuta = cargar_silver_ceuta()
print(f"Demanda media: {ceuta[SERIE].mean():,.0f} kW · máxima: {ceuta[SERIE].max():,.0f} kW")

In [ ]:
import matplotlib.pyplot as plt

serie = ceuta.set_index("DateTime")[SERIE]
fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))
diaria = serie.resample("D").mean()
axes[0].plot(diaria.index, diaria / 1000, lw=0.8)
axes[0].set(title="Demanda media diaria de Ceuta", ylabel="MW")
mensual = serie.groupby(serie.index.month).mean()
axes[1].plot(mensual.index, mensual / serie.mean(), marker="o")
axes[1].axhline(1, color="gray", lw=0.8, ls="--")
axes[1].set(title="Perfil mensual (1 = promedio)", xlabel="Mes", xticks=range(1, 13))
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Validación: ¿cuánto histórico hace falta para un verano?

Para cada verano se entrena con cuatro historias que terminan el 30 de junio: `sin_verano` (enero a junio,
como en Tetuán), `1_anio`, `2_anios` y `3_anios`. Se compara el modelo en nivel con calendario anual (V0) con
el objetivo en razón sin calendario anual (V4) y con las tres referencias ingenuas.

In [ ]:
!python -m experimentos.ceuta.validacion

In [ ]:
import pandas as pd

from tetouan_power_forecast.medallon import raiz_datos

resultados = pd.read_csv(raiz_datos() / "reportes" / "ceuta" / "resultados_validacion.csv")
tabla = lambda df: df.pivot_table(index="historia", columns="metodo", values="MAPE", aggfunc="mean").round(2)
print("MAPE (%) medio de los veranos 2021 a 2026:")
display(tabla(resultados[resultados["verano"] != 2020]))
print("Verano de 2020 (confinamiento), aparte:")
display(tabla(resultados[resultados["verano"] == 2020]))

## Cómo leerlo

- **`sin_verano` frente a `1_anio`:** cuánto pierde un modelo que nunca vio un verano. Es la situación de la
  Zona 3 de Tetuán, así que mide cuánto podría mejorar con un año más de datos.
- **V0 frente a V4:** si el objetivo en razón sin calendario anual también ayuda en otro sistema, la
  recomendación de `docs/historico.md` no depende de Tetuán.
- **Contra las referencias ingenuas:** el criterio de éxito es el mismo que en Tetuán.

Los resultados quedan en `reportes/ceuta/resultados_validacion.csv`, en el Drive.